In [ ]:
!pip install monai lifelines scikit-survival shap SimpleITK tcia_utils -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/multimodal_prognosis/'

In [ ]:
from tcia_utils import nbia

# get all available public collections and print them
collections = nbia.getCollections()
print(collections)

In [ ]:
from tcia_utils import nbia

# get series using the exact collection name from the list
series_data = nbia.getSeries(collection="NSCLC-Radiomics")

print(f"Type: {type(series_data)}")
print(f"Total series: {len(series_data)}")
print(series_data[0])  

In [ ]:

start_idx = 1    
end_idx   = 200   

batch = series_data[start_idx:end_idx]
nbia.downloadSeries(
    batch,
    path='/content/drive/MyDrive/multimodal_prognosis/raw_ct/'
)
print(f"Batch {start_idx}–{end_idx} done.")

In [ ]:

import os
import numpy as np
import torch
import SimpleITK as sitk
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE   = Path('/content/drive/MyDrive/multimodal_prognosis/')
RAW    = BASE / 'raw_ct'
TENSOR = BASE / 'tensors'
TENSOR.mkdir(parents=True, exist_ok=True)

print("Mounted. BASE =", BASE)

In [ ]:
# see what folders exist
patient_dirs = sorted([d for d in RAW.iterdir() if d.is_dir()])
print(f"Patients found: {len(patient_dirs)}")
for p in patient_dirs:
    print(p.name)

In [ ]:
def find_dicom_series(root_dir):
    """Walk directory tree and return list of folders containing DICOMs."""
    series_dirs = []
    for dirpath, _, files in os.walk(root_dir):
        dcm_files = [f for f in files if f.endswith('.dcm')]
        if len(dcm_files) > 10:  # skip tiny folders
            series_dirs.append(dirpath)
    return sorted(series_dirs)

series_dirs = find_dicom_series(RAW)
print(f"DICOM series found: {len(series_dirs)}")
for s in series_dirs:
    print(s)

In [ ]:
TARGET_SIZE = (64, 64, 64)   # safe for free Colab RAM
HU_MIN, HU_MAX = -1000, 400  # lung window

def load_and_preprocess(dicom_dir, target_size=TARGET_SIZE):
    """Load a DICOM series, resample to target size, clip and normalise."""
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(dicom_dir)

    if len(dicom_names) == 0:
        return None

    reader.SetFileNames(dicom_names)
    image = reader.Execute()

    # resample to isotropic 1mm spacing first
    original_spacing = image.GetSpacing()
    original_size    = image.GetSize()

    new_spacing = [1.0, 1.0, 1.0]
    new_size = [
        int(round(original_size[i] * original_spacing[i] / new_spacing[i]))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(new_spacing)
    resampler.SetSize(new_size)
    resampler.SetInterpolator(sitk.sitkLinear)
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetOutputOrigin(image.GetOrigin())
    image = resampler.Execute(image)

    # resize to fixed target size
    resample2 = sitk.ResampleImageFilter()
    resample2.SetSize(target_size)
    resample2.SetOutputSpacing([
        image.GetSpacing()[i] * image.GetSize()[i] / target_size[i]
        for i in range(3)
    ])
    resample2.SetInterpolator(sitk.sitkLinear)
    resample2.SetOutputDirection(image.GetDirection())
    resample2.SetOutputOrigin(image.GetOrigin())
    image = resample2.Execute(image)

    # convert to numpy, clip HU, normalise to [0, 1]
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    arr = np.clip(arr, HU_MIN, HU_MAX)
    arr = (arr - HU_MIN) / (HU_MAX - HU_MIN)

    return arr  # shape: (D, H, W)

In [ ]:
# run this before the fixed Cell 6
import shutil
for f in TENSOR.glob('*.pt'):
    print(f"Deleting: {f.name}")
    f.unlink()

In [ ]:
processed = []
failed    = []

for i, series_path in enumerate(series_dirs):
    parts = Path(series_path).parts
    print(f"Series {i}: {parts}")  

    patient_id = f"patient_{i:03d}"  
    out_path   = TENSOR / f"{patient_id}.pt"

    if out_path.exists():
        print(f"  Skipping {patient_id} (already saved)")
        processed.append(patient_id)
        continue

    print(f"  Processing {patient_id}...", end=' ')
    arr = load_and_preprocess(series_path)

    if arr is None:
        print("FAILED")
        failed.append(patient_id)
        continue

    tensor = torch.tensor(arr).unsqueeze(0)
    torch.save(tensor, out_path)
    processed.append(patient_id)
    print(f"saved. Shape: {tensor.shape}")

print(f"\nDone. Processed: {len(processed)}, Failed: {len(failed)}")

In [ ]:
# load one saved tensor and visualise 3 orthogonal slices
sample_path = list(TENSOR.glob('*.pt'))[0]
vol = torch.load(sample_path).squeeze().numpy()  # (D, H, W)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(vol[vol.shape[0]//2, :, :], cmap='gray')
axes[0].set_title('Axial')
axes[1].imshow(vol[:, vol.shape[1]//2, :], cmap='gray')
axes[1].set_title('Coronal')
axes[2].imshow(vol[:, :, vol.shape[2]//2], cmap='gray')
axes[2].set_title('Sagittal')
plt.suptitle(sample_path.stem)
plt.tight_layout()
plt.savefig(BASE / 'sample_slices.png')
plt.show()
print(f"Volume shape: {vol.shape}, min: {vol.min():.3f}, max: {vol.max():.3f}")

In [ ]:
# synthetic survival labels for the 200 patients
np.random.seed(42)
patient_ids = [p.stem for p in sorted(TENSOR.glob('*.pt'))]

labels = pd.DataFrame({
    'patient_id'   : patient_ids,
    'survival_days': np.random.randint(100, 1500, size=len(patient_ids)),
    'event'        : np.random.randint(0, 2,    size=len(patient_ids)),  # 1=died, 0=censored
    'age'          : np.random.randint(45, 80,  size=len(patient_ids)),
    'stage'        : np.random.choice([1, 2, 3, 4], size=len(patient_ids)),
    'gender'       : np.random.choice([0, 1],        size=len(patient_ids)),
})

labels.to_csv(BASE / 'NSCLC-Radiomics-Lung1', index=False)
print(labels)